In [ ]:
!pip install gensim  # NOTA: o Gensim ainda usa uma versão desatualizada do Numpy (<2)
                     #        por isso, para funcionar, vai ser necessário fazer
                     #        o downgrade do numpy e REINICIAR o ambiente antes
                     #        de executar esse script

  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.3.1
    Uninstalling numpy-2.3.1:
      Successfully uninstalled numpy-2.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tsfresh 0.21.0 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 1.13.1 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


#Exercícios complementares

##Sentence Embeddings
Os sentence embeddings são uma técnica de representação utilizada no processamento de linguagem natural (NLP) que transforma frases ou parágrafos inteiros em vetores numéricos em um espaço multidimensional. Essa abordagem permite que algoritmos compreendam o significado global do texto, considerando as relações entre as palavras e a estrutura das frases, o que é fundamental para diversas aplicações de NLP, como classificação de textos e sumarização automática.

Uma das formas mais simples de se construir sentence embeddings é por meio da representação de frases inteiras como vetores numéricos, utilizando as representações vetoriais das palavras que compõem essas frases. Essa técnica é especialmente útil em tarefas de processamento de linguagem natural (NLP), pois permite capturar o significado semântico das frases de maneira mais eficaz:

1. Inicialmente, cada palavra em uma frase é convertida em um vetor de word embeddings. Esses vetores são gerados por modelos como Word2Vec, GloVe ou FastText, que representam palavras em um espaço vetorial, onde palavras com significados semelhantes estão próximas umas das outras.
2. Após obter os vetores correspondentes a cada palavra, a abordagem mais simples para criar um sentence embedding é calcular a média desses vetores. Isso envolve somar todos os vetores das palavras da frase e, em seguida, dividir pelo número total de palavras. Essa média resulta em um vetor que representa a frase como um todo, capturando a semântica global.

Nesse sentido, vamos explorar o uso de Sentence Embeddings para realizar tarefas de classificação de documentos.



##Carregamento dos embeddings pré-treinados e do dataset
Executem o código a seguir para fazer o carregamento de Embeddings pré-treinados do website do NILC e fazer também o carregamento do dataset de análise de sentimentos.

In [ ]:
import io, os, zipfile, requests


EMBEDDING_DIM = 50

zip_url = f'https://www.dropbox.com/scl/fi/womf2nddm7605o2cox2pn/skip_50d.zip?rlkey=zidoab15bmpfmb9rgvc99vapg&st=4l8hoorw&dl=0'
response = requests.get(zip_url)

if (os.path.isfile(f'./skip_{EMBEDDING_DIM}d.txt')):
  print('Arquivo já existente no Runtime... Tudo OK')
else:
  if response.status_code == 200:
      with zipfile.ZipFile(io.BytesIO(response.content), 'r') as zip_ref:
          zip_ref.extractall('./')
          print("Download concluído e arquivo pronto...")
  else:
      print("Failed to download the zip file.")

Arquivo já existente no Runtime... Tudo OK


In [ ]:
import requests, os


filename = 'buscape.csv'

def download (filename):
  url = f'https://raw.githubusercontent.com/watinha/nlp-text-mining-datasets/main/{filename}'

  if (os.path.isfile(f'./{filename}')):
    print('Arquivo já existente no Runtime... Tudo OK')
    return

  response = requests.get(url)
  with open(f'./{filename}', 'wb') as f:
      f.write(response.content)
      print('Download realizado e arquivo extraído no Runtime... Tudo OK')

download(filename)



Arquivo já existente no Runtime... Tudo OK


##Extração de características

Implementem uma função que realize a extração de características dos documentos do córpus, calculando Sentence Embeddings, conforme descrito anteriormente. Nessa etapa, vocês podem implementar também uma rotina de pré-processamento, removendo stopwords e executando a lematização dos termos.

In [ ]:
import pandas as pd


df = pd.read_csv('buscape.csv').dropna()
corpus = df['review_text']
y = df['polarity']

In [ ]:
#!pip install spacy
!python -m spacy download pt_core_news_sm


In [ ]:
import spacy


nlp = spacy.load("pt_core_news_sm")

def preprocess(corpus, nlp):
  preprocessed_corpus = []

  for doc in corpus:
    doc = nlp(doc)
    tokens = [token.lemma_ for token in doc if not token.is_stop]
    preprocessed_corpus.append(' '.join(tokens))

  return preprocessed_corpus


preprocessed_corpus = preprocess(corpus, nlp)

In [ ]:
import numpy as np
from gensim.models import KeyedVectors


keyedvectors = KeyedVectors.load_word2vec_format(f'skip_s{EMBEDDING_DIM}.txt')

def extract_features(preprocessed_corpus, nlp, keyedvectors):
  X = []
  for doc in preprocessed_corpus:
    doc_embedding = np.zeros(EMBEDDING_DIM)
    num_words = 0
    doc = nlp(doc)
    for token in doc:
      if keyedvectors.has_index_for(token.text):
        doc_embedding += keyedvectors.get_vector(token.text)
        num_words += 1
    if num_words > 0:
      doc_embedding /= num_words
    X.append(doc_embedding)
  return np.array(X)


X_emb = extract_features(preprocessed_corpus, nlp, keyedvectors)



##Treinamento e avaliação
Separe o dataset em treinamento e teste (com um random_state=42) e realize a avaliação de modelos gerados com Árvores de Decisão, Random Forest, Logistic Regression, KNN e SVM.

Analisem como aumentar a quantidade de dimensões (50, 300, 1000) dos embeddings altera os resultados dessa abordagem de classificação de documentos.

**Resposta:** ao aumentar o número de dimensões os classificadores devem apresentar um resultado melhor, mas ainda inferior aos modelos com base em BOW. Isso sem utilizar otimização de hyper parâmetros.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

X_train, X_test, y_train, y_test = train_test_split(X_emb, y, random_state=42)
models = {
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(),
    'Logistic Regression': LogisticRegression(),
    'KNN': KNeighborsClassifier(),
    'SVM': SVC()
}

for model_name, model in models.items():
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  print(f'\n - {model_name}:')
  print(classification_report(y_test, y_pred))




 - Decision Tree:
              precision    recall  f1-score   support

         0.0       0.25      0.30      0.27      1705
         1.0       0.93      0.91      0.92     16702

    accuracy                           0.85     18407
   macro avg       0.59      0.60      0.60     18407
weighted avg       0.86      0.85      0.86     18407


 - Random Forest:
              precision    recall  f1-score   support

         0.0       0.84      0.12      0.21      1705
         1.0       0.92      1.00      0.96     16702

    accuracy                           0.92     18407
   macro avg       0.88      0.56      0.58     18407
weighted avg       0.91      0.92      0.89     18407


 - Logistic Regression:
              precision    recall  f1-score   support

         0.0       0.75      0.30      0.43      1705
         1.0       0.93      0.99      0.96     16702

    accuracy                           0.93     18407
   macro avg       0.84      0.65      0.69     18407
weighted av

##Clustering com Sentence Embeddings
Além de tarefas de classificação, nós também podemos utilizar os sentence embeddings para realizar tarefas de clusterização. Faça a clusterização dos documentos de sentimento **negativo** em **100 clusters** com embeddings e utilize um texto de teste a seguir para verificar documentos que pertencem ao mesmo cluster.

**Resposta:**
- esse celular ,tem muitos deficiencias ,bateria ruim não recomendo...
- Comprei por causa da bateria que prometia durar 900 horas mais ou menos, mas me arrependi pois dura só 2 dias...
- Deveria apresentar bateria de melhor qualidade, ...
- Bonito aparelho mais só é bom pra quem andar com carregador 24h ao lado...
- o produto não é recomendado para quem quer usar constantemente, pois a bateria dura pouco,...

In [ ]:
from sklearn.cluster import KMeans


df['features'] = X_emb.tolist()
negative_corpus = df[df['polarity'] == 0]['review_text'].tolist()
X_neg_emb = df[df['polarity'] == 0]['features'].tolist()

model = KMeans(n_clusters=100)
model.fit(X_neg_emb)

test = "Não comprem esse celular. Não funciona bem, a bateria não dura nada e a tela é monocromática."


preproccessed_test = preprocess([test], nlp)
test_embedding = extract_features(preproccessed_test, nlp, keyedvectors)
[cluster_id] = model.predict(test_embedding)

clusters_df = pd.DataFrame()
clusters_df['text'] = negative_corpus
clusters_df['cluster'] = model.labels_

doc_cluster = clusters_df.loc[clusters_df['cluster'] == cluster_id]
print('\n \n --- \n \n'.join(doc_cluster['text'].head().tolist()))

esse celular ,tem muitos deficiencias ,bateria ruim não recomendo

O que gostei: praticidade pelo tamanho

O que não gostei: bateria ruim, descascando o niquelado externo, tremor interno na chamada como se tudo estivesse solto e a partir de hoje o Touchscreen nao funciona, p,essimo, cuidado ao comprar
 
 --- 
 
Possui uma excelente câmera e é muito fácil manuseá-lo.

A maior dificuldade que encontro com esse aparelho é que a bateria não dura por mais de meia hora de uso.

Acessar as redes sociais também é um problema, quase nunca funciona.
 
 --- 
 
Comprei por causa da bateria que prometia durar 900 horas mais ou menos, mas me arrependi pois dura só 2 dias.

O que gostei: Preço bem popular

O que não gostei: A bateria não dura mais que 2 dias. É mentira que ela dure 900 horas.
 
 --- 
 
Deveria apresentar bateria de melhor qualidade, tendo em vista que pelo menos o meu não dura o suficiente para assistir a uma partida de futebol.

O que gostei: Tamanho.

O que não gostei: Duração da b